<a href="https://colab.research.google.com/github/suvigyajain0101/Large-Audio-Model-LAM/blob/main/JAX_Foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01: JAX Foundations for Large Audio Models (LAMs)

Here we build a foundation in **research-grade JAX**.

---
## Module 1 (Refresher): Functional Foundations

To scale seamlessly across GPUs and TPUs via the **XLA (Accelerated Linear Algebra)** compiler, JAX enforces three functional programming rules:

1. **Immutable Arrays (`.at[].set()`)**:
   In NumPy/PyTorch, `x[0] = 5` mutates memory in place. In JAX, arrays are immutable. You express updates functionally via `y = x.at[idx].set(val)` (or `.add()`, `.multiply()`). Inside `@jax.jit`, XLA's memory aliasing pass automatically compiles this into an in-place hardware buffer update whenever `x` is no longer needed.
2. **Pure Functions**:
   Every JIT-compiled or transformed function must depend *only* on its explicit arguments and produce *zero side effects* (no mutating globals).
3. **Explicit PRNG State (`jax.random.split`)**:
   Instead of a hidden global random seed, JAX passes an explicit `key = jax.random.PRNGKey(seed)`. Every time you need randomness, you **split** the key (`key, subkey = jax.random.split(key)`) and pass `subkey` as the **first argument** to the random op.

In [3]:
import jax
import jax.numpy as jnp

# 1.1 Vectorized One-Hot Encoding
# In Audio Tokenizers (VQ / BEST-RQ), we map discrete codebook IDs (e.g. [3, 0, 2, 1])
# into one-hot vectors using functional .at[row_indices, col_indices].set(1.0)
def custom_one_hot(labels: jax.Array, num_classes: int) -> jax.Array:
    """Vectorized one-hot encoding using functional .at[row, col].set(1.0)."""
    init_array = jnp.zeros(shape=(len(labels), num_classes), dtype=jnp.float32)
    row_indices = jnp.arange(len(labels))
    return init_array.at[row_indices, labels].set(1.0)


# 1.2 Stateless Inverted Dropout with Explicit PRNG Key Splitting
def dropout_layer(x: jax.Array, key: jax.Array, drop_rate: float = 0.5) -> tuple[jax.Array, jax.Array]:
    """Applies inverted dropout and returns (scaled_output, next_key)."""
    mask_key, next_key = jax.random.split(key, 2)
    keep_prob = 1.0 - drop_rate
    keep_mask = jax.random.bernoulli(mask_key, p=keep_prob, shape=x.shape).astype(x.dtype)
    scaled_x = (x * keep_mask) / keep_prob
    return scaled_x, next_key


# Quick Verification
audio_codebook_ids = jnp.array([3, 0, 2, 1])
one_hot_codes = custom_one_hot(audio_codebook_ids, num_classes=4)
print("One-hot encoded audio tokens:\n", one_hot_codes)

rng_key = jax.random.PRNGKey(42)
sample_features = jnp.ones((4, 8))
dropped_features, rng_key = dropout_layer(sample_features, rng_key, drop_rate=0.25)
print("\nDropped features:\n", dropped_features)
print("Updated PRNG key:", rng_key)

One-hot encoded audio tokens:
 [[0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]]

Dropped features:
 [[1.3333334 1.3333334 0.        1.3333334 1.3333334 1.3333334 1.3333334
  1.3333334]
 [1.3333334 1.3333334 1.3333334 1.3333334 0.        1.3333334 1.3333334
  1.3333334]
 [1.3333334 1.3333334 1.3333334 0.        1.3333334 1.3333334 1.3333334
  0.       ]
 [1.3333334 0.        0.        0.        0.        1.3333334 1.3333334
  1.3333334]]
Updated PRNG key: [  64467757 2916123636]


---
## Module 2 (Refresher): Core Transformations (`@jit`, `vmap`, `value_and_grad`)

1. **`@jax.jit(static_argnames=[...])`**:
   Traces your function using abstract `Tracer` objects (which carry `shape` and `dtype`, but *not* concrete runtime values) and compiles it into fused XLA machine code. Any argument used in a Python `if` condition or to determine array shapes must be listed in `static_argnames`.
2. **`jax.vmap(fn, in_axes=...)`**:
   Write your function for a **single example** (e.g., a single audio frame or a single query vector), and let `jax.vmap` vectorize it across a batch automatically. Use `0` for batched inputs and `None` for shared parameters/matrices.
3. **`jax.value_and_grad(loss_fn, argnums=0)`**:
   Computes both the scalar loss and parameter gradients $\nabla_w \mathcal{L}(w)$ in a single fused forward-backward pass.

In [5]:
# 2.1 Automatic Vectorization (vmap): Single-Frame Query Attention -> Multi-Frame Attention
# Suppose our Conformer outputs 25 audio frames per second (1 frame every 40ms), each of dimension d_model=64.
def compute_query_score(q_single: jax.Array, K: jax.Array) -> jax.Array:
    """Computes scaled dot-product attention scores for ONE query (d_model,) against K (seq_len, d_model)."""
    d_model = q_single.shape[-1]
    return jnp.dot(K, q_single) / jnp.sqrt(d_model)

# Vectorize over axis 0 of Q (num_queries, d_model), keeping K (seq_len, d_model) unbatched (None)
batched_compute_scores = jax.jit(jax.vmap(compute_query_score, in_axes=(0, None)))


# 2.2 JIT with Static Arguments & Single-Pass Value + Gradient
@jax.jit(static_argnames=["use_l2"])
def compute_regularized_loss(
    weights: jax.Array, X: jax.Array, y: jax.Array, use_l2: bool, reg_lambda: float = 0.01
) -> jax.Array:
    preds = jnp.dot(X, weights)
    mse_loss = jnp.mean((preds - y) ** 2)
    # Because `use_l2` is in static_argnames, Python can branch on it at trace time!
    if use_l2:
        l2_penalty = reg_lambda * jnp.sum(weights ** 2)
        return mse_loss + l2_penalty
    return mse_loss

# Differentiate with respect to argument 0 (`weights`)
loss_and_grad_fn = jax.value_and_grad(compute_regularized_loss, argnums=0)


# Quick Verification
Q_audio_frames = jnp.ones((25, 64))   # 25 audio frames (1 second at 40ms/frame), d_model=64
K_audio_frames = jnp.ones((25, 64))
attn_logits = batched_compute_scores(Q_audio_frames, K_audio_frames)
print("Attention logits shape (25 queries x 25 keys):", attn_logits.shape)

w_init = jnp.zeros((64,))
dummy_y = jnp.ones((25,))
loss_val, w_grads = loss_and_grad_fn(w_init, Q_audio_frames, dummy_y, True)
print(f"Initial Loss: {loss_val:.4f} | Grads shape: {w_grads.shape} | Grads norm: {jnp.linalg.norm(w_grads):.4f}")

Attention logits shape (25 queries x 25 keys): (25, 25)
Initial Loss: 1.0000 | Grads shape: (64,) | Grads norm: 16.0000
